# Spinor Band Structure & Local-Frame Unfolding for Real-Space Magnetic Textures

This notebook constructs tight-binding Hamiltonians in the presence of spatially varying magnetic textures (e.g. skyrmions, spirals) and performs **local-frame spin-resolved band unfolding** onto effective primitive Brillouin zone (pBZ) dispersion curves.

### Key Theoretical Foundations
1. **$SU(2)$ Rodrigues Spin Rotation**:
   The local orientation of magnetization $\mathbf{m}_i = (\sin\theta_i \cos\phi_i, \sin\theta_i \sin\phi_i, \cos\theta_i)$ is encoded via an $SU(2)$ rotation matrix $u_i$ satisfying $u_i (\mathbf{s}_{ref} \cdot \mathbf{\sigma}) u_i^\dagger = \mathbf{m}_i \cdot \mathbf{\sigma}$.
2. **Onsite Exchange Rotation vs. Gauge Invariance**:
   To ensure a physical energy dependence on magnetic misalignment ($L_{sk} \gg a$), local exchange splitting $\Delta_i$ is rotated in the local spin frame while kinetic hoppings $t_{ij} I_2$ remain fixed in the laboratory frame.
3. **Supercell $k$-Vector Scaling**:
   Because the real-space supercell is enlarged ($N_x \times N_y \times N_z$), supercell reciprocal units $\mathbf{B}^*$ are $1/N_i$ of primitive units $\mathbf{b}^*$. Primitive path points $\mathbf{k}_{prim}$ are scaled by $\mathbf{k}_{sc} = \mathbf{k}_{prim} \cdot \text{diag}([N_x, N_y, N_z])$.
4. **Momentum-Sector Filtered Unfolding**:
   To avoid folded BZ overlap ("spaghetti"), each point $\mathbf{k}_{prim}$ along the path selects its unique matching sector $iq = \text{argmin} \| \mathbf{k}_p(iq) - \mathbf{k}_{prim} \|$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pythtb import w90

from tb_spinor import *
from magn_spinor import *

import plotly.graph_objects as go


In [ ]:
# USER SETTINGS (collinear TB + plotting)
seed_up = "wannier90.1"
seed_dn = "wannier90.2"

w90_dir = "/Users/guymoore/Documents/ResearchProjects/magnetism/BiFeO3_TB/00_R3c_cl"

n_k_per_segment = 32
fermi_level = 4.50820836  # set to EF (eV) if you know it

# k-path settings (fractional reciprocal coords)
g = np.array([0.0, 0.0, 0.0])
k_delta = 0.1
s1 = np.array([-k_delta, 0.5, k_delta], dtype=float)
s2 = np.array([k_delta, 0.5, -k_delta], dtype=float)

k_nodes = [
    ("s1", s1),
    ("Γ", g),
    ("s2", s2),
]

# USER SETTINGS (texture + sU(2) assignment)
# Cone-plot visualization window (only for plotting the texture)
l_x = 1.0
n_x = 6
texture_z_plane = 0.0  # used only for the cone plot

# Texture parameters in the analytic formula
r0 = 1.0

# when assigning u(r) to wannier orbital centers:
# the skyrmion texture uses x,y in the *same units* as your orbital Cartesian coordinates.
# If your orbital centers are in angstrom, leave texture_scale=1.0; if not, tune it.
texture_scale = 1.0
texture_center_xy = (0.0, 0.0)  # (x_center, y_center) shift in Cartesian coordinates

# Reference spin direction for sU(2) rotations
# Our u(r) construction aligns the reference axis to n(r).
sref = np.array([0.0, 0.0, 1.0], dtype=float)

# supercell toggle:
# - None => rotate only within the reference cell (cell_key = (0,0,0))
# - not None => you must supply u for each (supercell internal cell, wf_index)

# sc_red_lat = None
# sc_red_lat = np.diag([1, 1, 1])
sc_red_lat = np.diag([n_x, 1, 1])
# sc_red_lat = np.diag([n_x, n_x, 1])

to_home = True

# pruning hoppings when importing wannier90 into PythTB
min_hopping_norm = 1.0e-2
# --------------------------
# MaGNETIC TEXTURE / INITIaLIZaTION sELECTION
# --------------------------

# # Skyrmion texture
# mag_fn = magnetization_texture_skyrmion_00

## Texture in one dimension (domain wall/cycloid)
f_x = lambda x: x / l_x  # custom 1D rotation profile
mag_fn = lambda x, y, z=0.0, r0=1.0: magnetization_texture_1d(x, y=y, z=z,
  r0=r0, f_fn=f_x, plane='yz')

# ## Uniform Collinear Magnetization along custom 3D vector vec=[mx, my, mz]
# # mag_fn = lambda x, y, z=0.0, r0=1.0: magnetization_collinear(x, y, z=z, r0=r0, vec=[1.0, 0.0, 0.0])  # +x
# mag_fn = lambda x, y, z=0.0, r0=1.0: magnetization_collinear(x, y, z=z, r0=r0, vec=[0.0, 1.0, 0.0])  # +y
# # mag_fn = lambda x, y, z=0.0, r0=1.0: magnetization_collinear(x, y, z=z, r0=r0, vec=[0.0, 0.0, 1.0])  # +z


In [ ]:
# --------------------------
# Helper: build a k-path (same style as your collinear script)
# --------------------------
def build_kpath(k_nodes, n_per_segment):
    labels = [k[0] for k in k_nodes]
    kpts = np.array([k[1] for k in k_nodes], dtype=float)

    k_list = []
    x_list = []
    tick_positions = [0.0]
    x = 0.0

    for i in range(len(kpts) - 1):
        k0 = kpts[i]
        k1 = kpts[i + 1]
        for j in range(n_per_segment):
            t = j / float(n_per_segment)
            k = (1 - t) * k0 + t * k1
            k_list.append(k)
            x_list.append(x)
            if j < n_per_segment - 1:
                dk = np.linalg.norm((k1 - k0) / n_per_segment)
                x += dk
        tick_positions.append(x)

    k_list.append(kpts[-1])
    x_list.append(x)

    return np.array(k_list), np.array(x_list), labels, np.array(tick_positions)


In [ ]:
k_vec, x_vec, labels, tick_pos = build_kpath(k_nodes, n_k_per_segment)


## 2. Real-Space Magnetic Texture Visualization
Visualize the real-space vector orientation $\mathbf{m}_i = (m_x, m_y, m_z)$ across the supercell grid using 3D vector fields.


In [ ]:
# grids
nx, ny = sc_red_lat[0, 0], sc_red_lat[1, 1]
spacing_scale = 1.5
L = 0.2  # full arrow length in data units

x_phys = np.linspace(-0.5*l_x, 0.5*l_x, nx, endpoint=False)
y_phys = np.linspace(-0.5*l_x, 0.5*l_x, ny, endpoint=False)
xg_phys, yg_phys = np.meshgrid(x_phys, y_phys, indexing="xy")

x_plot = np.linspace(-0.5*l_x*spacing_scale, 0.5*l_x*spacing_scale, nx, endpoint=False)
y_plot = np.linspace(-0.5*l_x*spacing_scale, 0.5*l_x*spacing_scale, ny, endpoint=False)
xg_plot, yg_plot = np.meshgrid(x_plot, y_plot, indexing="xy")
zg = np.full_like(xg_plot, texture_z_plane, dtype=float)

# directions from physical coordinates
n_grid = mag_fn(xg_phys, yg_phys, z=texture_z_plane, r0=r0)
u = n_grid[..., 0].ravel()
v = n_grid[..., 1].ravel()
w = n_grid[..., 2].ravel()

# centers
xc, yc, zc = xg_plot.ravel(), yg_plot.ravel(), zg.ravel()

# normalize directions
norm = np.sqrt(u*u + v*v + w*w) + 1e-15
ux, vy, wz = u/norm, v/norm, w/norm

# centered endpoints: start = center - (L/2) n, end = center + (L/2) n
h = 0.5 * L
x0, y0, z0 = xc - h*ux, yc - h*vy, zc - h*wz
x1, y1, z1 = xc + h*ux, yc + h*vy, zc + h*wz

# line segments with None separators
xl, yl, zl = [], [], []
for i in range(len(xc)):
    xl += [x0[i], x1[i], None]
    yl += [y0[i], y1[i], None]
    zl += [z0[i], z1[i], None]

fig = go.Figure()

# main arrows
fig.add_trace(go.Scatter3d(
    x=xl, y=yl, z=zl,
    mode="lines",
    line=dict(color="blue", width=12),
    showlegend=False
))
fig.add_trace(go.Scatter3d(
    x=x1, y=y1, z=z1,
    mode="markers",
    marker=dict(color="blue", size=9),
    showlegend=False
))

# --------- add XYZ compass (small triad) ----------
# place near lower-left of data bounds
xmin, xmax = np.min(xg_plot), np.max(xg_plot)
ymin, ymax = np.min(yg_plot), np.max(yg_plot)
zmin, zmax = np.min(zg), np.max(zg)
zr = max(1e-6, zmax - zmin)

origin = np.array([
    xmin - 0.15*(xmax - xmin),
    ymin - 0.15*(ymax - ymin),
    zmin + 0.15*zr
], dtype=float)

l_comp = 0.05 * max((xmax - xmin), (ymax - ymin), zr)

axes = [
    ("x", np.array([1,0,0], float), "red"),
    ("y", np.array([0,1,0], float), "green"),
    ("z", np.array([0,0,1], float), "black"),
]

for lab, d, col in axes:
    p1 = origin + l_comp*d
    # line
    fig.add_trace(go.Scatter3d(
        x=[origin[0], p1[0]],
        y=[origin[1], p1[1]],
        z=[origin[2], p1[2]],
        mode="lines",
        line=dict(color=col, width=10),
        showlegend=False
    ))
    # tip marker + label
    fig.add_trace(go.Scatter3d(
        x=[p1[0]], y=[p1[1]], z=[p1[2]],
        mode="markers+text",
        marker=dict(color=col, size=5),
        text=[lab],
        textposition="top center",
        textfont=dict(size=14, color=col),
        showlegend=False
    ))

# hide axes completely
fig.update_layout(
    width=900, height=700,
    margin=dict(l=0, r=0, t=0, b=0),
    scene=dict(
        # camera_projection_type="orthographic",
        aspectmode="data",
        xaxis=dict(visible=False, showgrid=False, zeroline=False, showticklabels=False, title=""),
        yaxis=dict(visible=False, showgrid=False, zeroline=False, showticklabels=False, title=""),
        zaxis=dict(visible=False, showgrid=False, zeroline=False, showticklabels=False, title=""),
    ),
)

fig.update_layout(
    scene=dict(
        camera_projection_type="perspective",
        camera=dict(
            eye=dict(x=4, y=4, z=4),  # farther back than ~1,1,1
            center=dict(x=0, y=0, z=0),
            up=dict(x=0, y=0, z=1),
        ),
    )
)

fig.show()

In [ ]:
# # ---------- controls ----------
# spacing_scale = 2.8   # only affects where cones are drawn
# # -----------------------------

# nx = sc_red_lat[0, 0]
# ny = sc_red_lat[1, 1]

# # physical coordinates (for mag_fn) -- unchanged
# x_phys = np.linspace(-0.5 * l_x, 0.5 * l_x, nx, endpoint=False)
# y_phys = np.linspace(-0.5 * l_x, 0.5 * l_x, ny, endpoint=False)
# xg_phys, yg_phys = np.meshgrid(x_phys, y_phys, indexing="xy")

# # plotting coordinates (for display) -- expanded
# x_plot = np.linspace(-0.5 * l_x * spacing_scale, 0.5 * l_x * spacing_scale, nx, endpoint=False)
# y_plot = np.linspace(-0.5 * l_x * spacing_scale, 0.5 * l_x * spacing_scale, ny, endpoint=False)
# xg_plot, yg_plot = np.meshgrid(x_plot, y_plot, indexing="xy")

# zg = texture_z_plane * np.ones_like(xg_plot)

# # evaluate texture on physical grid
# n_grid = mag_fn(xg_phys, yg_phys, z=texture_z_plane, r0=r0)
# mx, my, mz = n_grid[..., 0], n_grid[..., 1], n_grid[..., 2]

# fig = go.Figure(
#     data=go.Cone(
#         x=xg_plot.ravel(), y=yg_plot.ravel(), z=zg.ravel(),   # display positions
#         u=mx.ravel(), v=my.ravel(), w=mz.ravel(),             # physical directions
#         colorscale=[[0, "blue"], [1, "blue"]],
#         showscale=False,
#         sizemode="absolute",
#         sizeref=0.6,
#         anchor="tail",
#     )
# )

# fig.update_layout(
#     width=900, height=700,
#     margin=dict(l=0, r=0, t=0, b=0),
#     scene=dict(
#         camera_projection_type="orthographic",
#         aspectmode="data",
#         zaxis=dict(visible=False),
#     ),
# )
# fig.show()

## 3. Load Wannier90 Models & Build Spinful Hamiltonian
Combine spin-up and spin-down Wannier models into a $2N_{orb} \times 2N_{orb}$ spinful tight-binding model.


In [ ]:
# --------------------------
# Load wannier90 + build PythTB models
# --------------------------
w90_up = w90(w90_dir, seed_up)
w90_dn = w90(w90_dir, seed_dn)

print("Creating TB model...")

tb_up = w90_up.model(min_hopping_norm=min_hopping_norm)
tb_dn = w90_dn.model(min_hopping_norm=min_hopping_norm)

print("solving TB model... (texture rotation happens next)")


In [ ]:
orb_up = tb_up.get_orb()
orb_dn = tb_dn.get_orb()

diff_raw = orb_up - orb_dn
diff_wrapped = diff_raw - np.round(diff_raw)  # wrap into ~[-0.5,0.5]

print("max raw  |diff|      =", np.max(np.abs(diff_raw)))
print("max wrapped |diff|  =", np.max(np.abs(diff_wrapped)))


In [ ]:
tb_spinful = build_spinful_from_collinear_intersection(tb_up, tb_dn, fermi_level=fermi_level)

dim_r = tb_spinful._dim_r
base_norb = tb_spinful._norb

print(f"spinful model: dim_r={dim_r}, base_norb={base_norb}, nspin={tb_spinful._nspin}")


## 4. Construct $SU(2)$ Spin-Rotation Matrices
Compute $u_i \in SU(2)$ for every Wannier orbital center in the supercell and enforce sign-gauge continuity across cells.


In [ ]:
# --------------------------
# Build the supercell geometry if needed
# --------------------------
if sc_red_lat is not None:
    sc_tb, sc_vectors = tb_spinful.make_supercell(
        sc_red_lat, return_sc_vectors=True, to_home=to_home
    )
else:
    sc_tb = tb_spinful
    sc_vectors = [np.zeros(dim_r, dtype=int)]

lat_sc = sc_tb.get_lat()
orb_sc = sc_tb.get_orb()  # reduced coords for each orbital in the sc model

# sanity check
num_sc = len(sc_vectors)
if sc_tb._norb != base_norb * num_sc:
    raise ValueError(
        f"Unexpected supercell orbital count: sc_tb._norb={sc_tb._norb}, "
        f"expected base_norb*num_sc={base_norb*num_sc}"
    )

u_samples = {}
x_center, y_center = texture_center_xy

for sc_i, cell_r in enumerate(sc_vectors):
    cell_key = tuple(int(x) for x in np.asarray(cell_r, dtype=int))

    for wf_i in range(base_norb):
        orb_i = sc_i * base_norb + wf_i

        # Cartesian position of wannier orbital center:
        # orb_sc[orb_i] are reduced coordinates in dim_r; lat_sc maps to Cartesian.
        r_cart = np.dot(orb_sc[orb_i], lat_sc)  # (dim_r,)

        x_cart = float(r_cart[0])
        y_cart = float(r_cart[1])
        z_cart = float(r_cart[2]) if dim_r >= 3 else 0.0

        # shift & scale before evaluating texture
        x_tex = texture_scale * (x_cart - x_center)
        y_tex = texture_scale * (y_cart - y_center)

        n_loc = np.squeeze(mag_fn(x_tex, y_tex, z=z_cart, r0=r0))

        u_loc = su2_from_ref_to_n(sref=sref, n=n_loc)  # (2,2)

        u_samples[(cell_key, int(wf_i))] = u_loc

print(f"Built u_samples with {len(u_samples)} entries.")

# Optional: sU(2) sign continuity gauge fix
# (Does not affect eigenvalues; can help if you later use eigenvectors for Berry phases.)
try:
    ordered_keys = sorted(u_samples.keys())
    u_stack = np.array([u_samples[k] for k in ordered_keys], dtype=complex)
    u_stack_fixed = su2_fix_sign_continuity(u_stack)
    for k, u in zip(ordered_keys, u_stack_fixed):
        u_samples[k] = u
        # print(u)
    print("applied sU(2) sign continuity fix.")
except Exception as e:
    print(f"skipping sign continuity fix due to: {e}")


## 5. Solve Rotated Supercell Hamiltonian
Evaluate supercell eigenvalues $e_{sc}(k)$ and eigenvectors $v_{sc}(k)$ across the scaled supercell $k$-path $\mathbf{k}_{sc} = \mathbf{k}_{prim} \cdot \text{diag}([N_x, N_y, N_z])$.


In [ ]:
# Convert primitive k-path to supercell reciprocal units:
k_sc_input = k_vec * np.diag(sc_red_lat)

import time

# ---------------------------------------------------------------------------
# 1. Multi-Process + Numba Parallel Solver (Optimized, Default)
# ---------------------------------------------------------------------------
t0 = time.perf_counter()
e_rot, evecs_rot = compute_rotated_bands_parallel(
    tb_up, tb_dn, k_sc_input, u_samples,
    sc_red_lat=sc_red_lat,
    eig_vectors=True,
    n_jobs=-1
)
t_par = time.perf_counter() - t0

# # ---------------------------------------------------------------------------
# # 2. Unoptimized Serial Reference Solver (For Timing & Verification)
# # ---------------------------------------------------------------------------
# t0 = time.perf_counter()
# e_rot_ser, evecs_rot_ser = compute_rotated_bands_serial(
#     tb_up, tb_dn, k_sc_input, u_samples,
#     sc_red_lat=sc_red_lat,
#     eig_vectors=True
# )
# t_ser = time.perf_counter() - t0

# diff_e = np.max(np.abs(e_rot - e_rot_ser))
# speedup = t_ser / t_par if t_par > 0 else 1.0

# print(f"Parallel Solver Time : {t_par:.3f} s")
# print(f"Serial Solver Time   : {t_ser:.3f} s")
# print(f"Parallel Speedup     : {speedup:.2f}x")
# print(f"Max Eigenvalue Diff  : {diff_e:.2e} eV (Machine Precision!)")


In [ ]:
print("Bands computed.")
print("e_rot shape:", e_rot.shape)


In [ ]:
plt.figure(dpi=300, figsize=(4.5, 4.5))

for n in range(e_rot.shape[0]):
    plt.plot(
        x_vec,
        e_rot[n, :]-fermi_level,
        linewidth=0.9,
        alpha=0.9,
        color="k",
        linestyle="-",
    )

for xp in tick_pos:
    plt.axvline(xp, linewidth=0.8, color="gray", alpha=0.7)

plt.xticks(tick_pos, labels)
plt.ylabel(r"Energy $E - E_F$ (eV)")
plt.xlabel("k-path")

if sc_red_lat is None:
    plt.title("spinor TB bands (texture): primitive-cell assignment")
else:
    plt.title(f"spinor TB bands (texture) \n supercell: {sc_red_lat.tolist()}")

plt.tight_layout()
plt.show()


## 6. Density of States (DOS) & Projected DOS (PDOS)
Compute the total and orbital-projected electronic density of states via Gaussian-broadened energy integration.


In [ ]:
# evals: (nb, nk), evecs: (nb, nk, nbasis)  [or (nk, nbasis, nb)]
dos_out = compute_total_and_projected_dos(
    evals=e_rot,
    evecs=evecs_rot,
    kpoints=k_vec,
    projector={"indices": [0,1,2,3,4,5,6,7,8,9]},   # project onto selected wannier basis indices
    sigma=0.02,
    n_energy=3000,
    return_components=True,
)

energy = -fermi_level+dos_out["energy"]
dos = dos_out["dos"]
pdos = dos_out["pdos"]


In [ ]:
plt.figure(figsize=(6,4), dpi=300)
plt.plot(energy, dos, lw=2, label="total dos", color="black")
# plt.plot(energy, pdos, lw=2, label="projected dos", color="tab:red")
plt.xlabel("Energy (ev)")
plt.ylabel("DOs (states / ev)")
# plt.title("total and projected density of states")
plt.legend(frameon=False)
plt.tight_layout()
plt.show()


## 7. Local-Frame Spin-Resolved Band Unfolding
Project supercell wavefunctions onto local-frame reference primitive states $\tilde{C}^J_{(\mu,s,\mathbf{n})} = \sum_{s'} (u_i^\dagger)_{ss'} C^J_{(\mu,s',\mathbf{n})}$ and compute the momentum-filtered spectral function $a(k,e)$ and spin polarization $a_{xyz}(k,e)$.


In [ ]:
def gaussian_spectral_from_bands(
    E,              # (nb, nk)
    w,              # (nb, nk)  scalar spectral weight
    s=None,         # (nb, nk, 3) spin weights; optional
    e_grid=None,
    e_min=None,
    e_max=None,
    nE=600,
    sigma=0.03
):
    E = np.asarray(E, float)
    w = np.asarray(w, float)
    nb, nk = E.shape

    if e_grid is None:
        if e_min is None: e_min = E.min() - 5*sigma
        if e_max is None: e_max = E.max() + 5*sigma
        e_grid = np.linspace(e_min, e_max, nE)
    else:
        e_grid = np.asarray(e_grid, float)
        nE = e_grid.size

    a = np.zeros((nk, nE), float)
    axyz = np.zeros((nk, nE, 3), float) if s is not None else None

    if s is not None:
        s_abs = np.abs(np.asarray(s, float))

    norm = 1.0/(np.sqrt(2*np.pi)*sigma)

    for ik in range(nk):
        for jb in range(nb):
            g = norm*np.exp(-0.5*((e_grid - E[jb,ik])/sigma)**2)  # (nE,)
            a[ik] += w[jb,ik]*g
            if s is not None:
                axyz[ik,:,0] += s_abs[jb,ik,0]*g
                axyz[ik,:,1] += s_abs[jb,ik,1]*g
                axyz[ik,:,2] += s_abs[jb,ik,2]*g

    return e_grid, a, axyz


In [ ]:
from unfold_spin_project import *

proj = project_sc_bands_on_reference(
    tb_up=tb_up,
    tb_dn=tb_dn,
    e_sc=e_rot,
    v_sc=evecs_rot,
    k_s_list=k_sc_input,
    sc_red_lat=sc_red_lat,
    fermi_level=fermi_level,
    nb_ref_keep=2*tb_up._norb
)

# Momentum-filtering: pick the unique matching sector iq for each k-point ik along the path:
nband_sc, nk = e_rot.shape
w_pick = np.zeros((nband_sc, nk))
s_pick = np.zeros((nband_sc, nk, 3))

for ik in range(nk):
    k_p_prim = proj["k_p"][ik] / np.diag(sc_red_lat)
    diffs = np.linalg.norm(k_p_prim - k_vec[ik], axis=-1)
    iq_match = np.argmin(diffs)
    w_pick[:, ik] = proj["weights"][:, ik, iq_match, :].sum(axis=-1)
    s_pick[:, ik, :] = proj["spin_xyz"][:, ik, iq_match, :, :].sum(axis=-2)

w_iq = w_pick
s_iq = s_pick


In [ ]:
# Example: picked per-(band,k) values
# w_pick: (nb,nk), s_pick: (nb,nk,3), e_rot: (nb,nk)
e_grid, a, axyz = gaussian_spectral_from_bands(
    E=e_rot - fermi_level,
    w=w_iq,
    s=s_iq,
    sigma=0.03,
    nE=700
)


In [ ]:
eps = 1e-12
rgb = np.abs(axyz)                              # (nk,nE,3)
rgb /= np.maximum(np.sum(rgb, axis=2, keepdims=True), eps)  # normalize color
intensity = a / (a.max() + eps)                 # 0..1
img = np.clip(rgb * intensity[...,None], 0, 1)  # (nk,nE,3)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(5,4), dpi=250)
# imshow expects (Ny,Nx,3): Ny=nE, Nx=nk
plt.imshow(
    np.transpose(img, (1,0,2)),
    origin="lower",
    aspect="auto",
    extent=[x_vec.min(), x_vec.max(), e_grid.min(), e_grid.max()]
)

for xp in tick_pos:
    plt.axvline(xp, color='w', lw=0.6, alpha=0.4)
plt.xticks(tick_pos, labels)
plt.ylabel(r"$E - E_F$ (eV)")
plt.xlabel(r"$k$-path")
plt.tight_layout()
plt.show()
